# Etapa 2 - Pré-processamento

Split estratificado 60/20/20, imputação de ausentes, manutenção do encoding
numérico e scaling, tudo via `ColumnTransformer`, com `fit` apenas no treino
(prevenção de vazamento). Lógica em `src/preprocessing.py`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import config
from src import preprocessing as pp

config.set_seeds()

## 1. Quadro de escolhas e justificativas

Gera `outputs/metrics/preprocessing_choices.md` com cada decisão e sua justificativa.

In [2]:
from IPython.display import Markdown

path = pp.write_preprocessing_choices()
Markdown(path.read_text(encoding="utf-8"))

# Escolhas de Pré-processamento e Justificativas

| Aspecto | Escolha | Justificativa |
| --- | --- | --- |
| Split | Estratificado 60/20/20 (train/val/test) | Treino para ajuste; validação para early stopping/Optuna; teste avaliado uma única vez. Estratificação preserva a proporção das classes (e dos decis de BMI na regressão) nos três conjuntos, importante dado o desbalanceamento. |
| Prevenção de vazamento | `fit` apenas no treino | Imputador, scaler e (opcional) clipper aprendem estatísticas só do treino e são aplicados via `transform` em val/test, encapsulados em `ColumnTransformer`/`Pipeline`. |
| Valores ausentes (contínuas/ordinais) | `SimpleImputer(strategy='median')` | A mediana é robusta a outliers (relevantes em BMI/MentHlth/PhysHlth). O dataset não tem ausentes, mas a etapa cumpre o requisito e dá robustez a novas amostras. |
| Valores ausentes (binárias) | `SimpleImputer(strategy='most_frequent')` | Moda é a estatística adequada para variáveis 0/1. |
| Encoding | Manter codificação numérica original | Binárias já são 0/1; ordinais (`GenHlth`, `Age`, `Education`, `Income`) têm ordem natural — **não** se aplica One-Hot para não perder a ordem nem inflar a dimensão. |
| Scaling | `StandardScaler` em ordinais+contínuas; binárias `passthrough` | Padroniza escalas distintas (ex.: BMI ~12–98 vs. dias 0–30), ajudando a convergência da MLP. Padronizar variáveis 0/1 é desnecessário e prejudica a interpretabilidade, por isso passam direto. |
| Alternativa de scaling | `RobustScaler` (configurável) | Usa mediana/IQR; alternativa menos sensível aos outliers de BMI/MentHlth/PhysHlth. |
| Outliers (opcional) | Clipping de BMI no percentil 99 (flag `clip_bmi`) | Atenua a cauda extrema do BMI sem descartar amostras; limite aprendido no treino. Desativado por padrão; impacto documentado. |
| Desbalanceamento | `class_weight='balanced'` no treino (Etapa 4) | Compensa as classes minoritárias sem alterar os dados; complementado por métricas robustas (ROC-AUC, PR-AUC, F1 macro). |


## 2. Classificação das colunas por tipo

Binárias (passthrough) vs. ordinais + contínuas (imputação por mediana + scaling).

In [3]:
binary, ordinal, continuous = pp.classify_columns(config.ALL_FEATURES)
print("Binárias  :", binary)
print("Ordinais  :", ordinal)
print("Contínuas :", continuous)

Binárias  : ['HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex']
Ordinais  : ['GenHlth', 'Age', 'Education', 'Income']
Contínuas : ['BMI', 'MentHlth', 'PhysHlth']


## 3. Split estratificado 60/20/20 e pré-processamento

Exemplo na tarefa binária. O `fit` ocorre só no treino.

In [4]:
data = pp.prepare_data("binary")
print("X_train:", data.X_train.shape)
print("X_val  :", data.X_val.shape)
print("X_test :", data.X_test.shape)
print("features:", data.feature_names)

X_train: (152208, 21)
X_val  : (50736, 21)
X_test : (50736, 21)
features: ['HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex', 'GenHlth', 'Age', 'Education', 'Income', 'BMI', 'MentHlth', 'PhysHlth']


## 4. Verificação: sem vazamento e scaling correto

Numéricas com média ~0 e desvio ~1 no treino; no teste a média não é exatamente 0
(o scaler foi ajustado só no treino). Binárias permanecem 0/1.

In [5]:
names = data.feature_names
num_idx = [i for i, n in enumerate(names) if n in (config.ORDINAL_FEATURES + config.CONTINUOUS_FEATURES)]
bin_idx = [i for i, n in enumerate(names) if n in config.BINARY_FEATURES]

print("Numéricas — média (treino):", np.round(data.X_train[:, num_idx].mean(axis=0), 4))
print("Numéricas — desvio (treino):", np.round(data.X_train[:, num_idx].std(axis=0), 4))
print("Numéricas — média (teste) :", np.round(data.X_test[:, num_idx].mean(axis=0), 4))
print("Binárias  — valores únicos:", np.unique(data.X_train[:, bin_idx]))

Numéricas — média (treino): [ 0. -0. -0. -0.  0. -0. -0.]
Numéricas — desvio (treino): [1. 1. 1. 1. 1. 1. 1.]
Numéricas — média (teste) : [-0.0025 -0.0034  0.005   0.0075  0.0034 -0.0031 -0.0048]
Binárias  — valores únicos: [0. 1.]


## 5. Pesos de classe (para `class_weight` na Etapa 4)

In [6]:
print("binário    :", pp.compute_class_weights(data.y_train))
multi = pp.prepare_data("multiclass")
print("multiclasse:", pp.compute_class_weights(multi.y_train))

binário    : {0: 0.5809465648854962, 1: 3.5884571859675596}
multiclasse: {0: 0.39569181335350684, 1: 18.256926952141058, 2: 2.3923047906450394}


## 6. As três tarefas

Mesmo pipeline; a regressão remove `BMI` das features e mantém `Diabetes_binary` como preditor.

In [7]:
for task in ("binary", "multiclass", "regression"):
    d = pp.prepare_data(task)
    print(f"{task:11s} -> X_train={d.X_train.shape}, n_features={len(d.feature_names)}, alvo={d.target}")

binary      -> X_train=(152208, 21), n_features=21, alvo=Diabetes_binary
multiclass  -> X_train=(152208, 21), n_features=21, alvo=Diabetes_012
regression  -> X_train=(152208, 21), n_features=21, alvo=BMI


## 7. (Opcional) Alternativas configuráveis

`RobustScaler` e clipping de BMI no percentil 99 (comparáveis ao baseline).

In [8]:
d_robust = pp.prepare_data("binary", scaler="robust", clip_bmi=True)
bmi_i = d_robust.feature_names.index("BMI")
print("BMI (robust+clip) — min/max no treino:",
      round(float(d_robust.X_train[:, bmi_i].min()), 3),
      round(float(d_robust.X_train[:, bmi_i].max()), 3))

BMI (robust+clip) — min/max no treino: -2.143 3.143
